# Flash Attention from Scratch

Understanding Flash Attention: the IO-aware, exact attention algorithm that avoids materializing the full $O(n^2)$ attention matrix.

**Interview questions:**
- "Why is standard attention memory-inefficient?"
- "How does Flash Attention avoid materializing the full attention matrix?"
- "What is the online softmax trick?"
- "What is the difference between SRAM and HBM?"

---

## The Memory Problem

Standard attention: $\text{Attn}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

| Component | Shape | Memory for n=8192, d=128 |
|-----------|-------|-------------------------|
| Q, K, V | $(n, d)$ each | 3 MB each |
| $QK^T$ (scores) | $(n, n)$ | **256 MB** |
| softmax output | $(n, n)$ | **256 MB** |
| Output | $(n, d)$ | 3 MB |

The $O(n^2)$ attention matrix dominates memory. Flash Attention computes the **exact same result** without ever storing the full matrix.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import math
import time

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 1. Standard Attention Implementation

The naive implementation materializes the full $n \times n$ attention matrix. This is what we want to avoid.

In [ ]:
def standard_attention(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                       causal: bool = False) -> torch.Tensor:
    """Standard scaled dot-product attention.
    
    Args:
        Q: (batch, n, d)
        K: (batch, n, d)
        V: (batch, n, d)
        causal: whether to apply causal mask
    
    Returns:
        output: (batch, n, d)
    
    Memory: O(n^2) for the attention matrix
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute scores -- materializes full n x n matrix
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # (batch, n, n)
    
    # Step 2: Causal mask (optional)
    if causal:
        n = Q.shape[1]
        mask = torch.triu(torch.ones(n, n, device=Q.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
    
    # Step 3: Softmax -- another n x n matrix
    attn_weights = F.softmax(scores, dim=-1)  # (batch, n, n)
    
    # Step 4: Weighted sum
    output = torch.matmul(attn_weights, V)  # (batch, n, d)
    
    return output


# Test
B, N, D = 2, 64, 32
Q = torch.randn(B, N, D)
K = torch.randn(B, N, D)
V = torch.randn(B, N, D)

out = standard_attention(Q, K, V)
print(f"Input shapes: Q={Q.shape}, K={K.shape}, V={V.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention matrix would be: ({B}, {N}, {N}) = {B*N*N*4/1024:.1f} KB")

## 2. GPU Memory Hierarchy

Flash Attention exploits the GPU memory hierarchy:

```
    SRAM (on-chip)         HBM (off-chip)
    ┌─────────────┐        ┌─────────────────┐
    │  ~20 MB      │        │  ~40-80 GB       │
    │  ~19 TB/s    │  <-->  │  ~1.5-3 TB/s     │
    │  Very fast   │        │  10-20x slower    │
    └─────────────┘        └─────────────────┘
```

Standard attention reads/writes the full $n \times n$ matrix to HBM multiple times. Flash Attention keeps tiles in SRAM and accumulates results incrementally.

In [ ]:
# Visualize memory access patterns
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Standard attention: HBM round trips
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Standard Attention: Memory Access', fontsize=13)

# HBM box
hbm = mpatches.FancyBboxPatch((0.5, 0.5), 3.5, 9, boxstyle="round,pad=0.1",
                               facecolor='#ffcccc', edgecolor='black', linewidth=2)
ax.add_patch(hbm)
ax.text(2.25, 9.2, 'HBM', ha='center', fontweight='bold', fontsize=12)

# SRAM box  
sram = mpatches.FancyBboxPatch((6, 3), 3.5, 4, boxstyle="round,pad=0.1",
                                facecolor='#ccffcc', edgecolor='black', linewidth=2)
ax.add_patch(sram)
ax.text(7.75, 6.7, 'SRAM', ha='center', fontweight='bold', fontsize=12)

# Items in HBM
items = ['Q, K, V\n(3*n*d)', 'S = QK^T\n(n*n)', 'P = softmax(S)\n(n*n)', 'O = PV\n(n*d)']
for i, item in enumerate(items):
    y = 8 - i * 2
    ax.text(2.25, y, item, ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

# Arrows showing round trips
ax.annotate('', xy=(6, 5.5), xytext=(4, 7),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.annotate('', xy=(4, 5), xytext=(6, 4.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(5, 6.5, 'Read Q,K', fontsize=8, color='red')
ax.text(5, 4.2, 'Write S', fontsize=8, color='red')
ax.text(7.75, 5, 'Compute\nS = QK^T', ha='center', fontsize=9)
ax.text(5, 8.5, '4 HBM round trips!', fontsize=10, color='red', fontweight='bold')
ax.axis('off')

# Flash attention: tiled computation
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Flash Attention: Tiled Computation', fontsize=13)

# HBM
hbm2 = mpatches.FancyBboxPatch((0.5, 0.5), 3.5, 9, boxstyle="round,pad=0.1",
                                facecolor='#ffcccc', edgecolor='black', linewidth=2)
ax.add_patch(hbm2)
ax.text(2.25, 9.2, 'HBM', ha='center', fontweight='bold', fontsize=12)

# SRAM
sram2 = mpatches.FancyBboxPatch((6, 2), 3.5, 6, boxstyle="round,pad=0.1",
                                 facecolor='#ccffcc', edgecolor='black', linewidth=2)
ax.add_patch(sram2)
ax.text(7.75, 7.7, 'SRAM', ha='center', fontweight='bold', fontsize=12)

items2 = ['Q, K, V\n(3*n*d)', 'O (output)\n(n*d)']
for i, item in enumerate(items2):
    y = 7.5 - i * 3
    ax.text(2.25, y, item, ha='center', va='center', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', edgecolor='gray'))

# SRAM contents
sram_items = ['Q_tile (B_r x d)', 'K_tile (B_c x d)', 'V_tile (B_c x d)',
              'S_tile (B_r x B_c)', 'O_tile (B_r x d)']
for i, item in enumerate(sram_items):
    ax.text(7.75, 7 - i * 1.0, item, ha='center', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='gray'))

ax.annotate('', xy=(6, 6), xytext=(4, 7),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.annotate('', xy=(4, 4), xytext=(6, 3.5),
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.text(5, 6.8, 'Load tiles', fontsize=8, color='green')
ax.text(5, 3.2, 'Write O', fontsize=8, color='green')
ax.text(5, 8.5, 'No n*n matrix in HBM!', fontsize=10, color='green', fontweight='bold')
ax.text(2.25, 1.5, 'No S or P stored!', ha='center', fontsize=9, color='green',
        fontstyle='italic')
ax.axis('off')

plt.tight_layout()
plt.show()

## 3. The Online Softmax Trick

The key challenge: softmax requires knowing the max and sum over the **entire** row, but we process in tiles.

**Standard softmax:** $\text{softmax}(x_i) = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}$ where $m = \max(x)$

**Online softmax (Milakov & Gimelshein, 2018):** incrementally update $m$ and the denominator as new tiles arrive.

When we see a new tile with max $m_{\text{new}}$:
1. Update global max: $m' = \max(m_{\text{old}}, m_{\text{new}})$
2. Rescale old accumulator: multiply by $e^{m_{\text{old}} - m'}$
3. Add new tile contribution: scaled by $e^{m_{\text{new}} - m'}$

In [ ]:
def online_softmax_demo(x_full: torch.Tensor, block_size: int = 4):
    """Demonstrate online softmax: compute softmax in blocks without seeing all values at once.
    
    Args:
        x_full: (n,) vector to compute softmax over
        block_size: process this many elements at a time
    """
    n = x_full.shape[0]
    
    # Running statistics
    m = torch.tensor(float('-inf'))  # running max
    l = torch.tensor(0.0)           # running sum of exp
    
    # Process in blocks
    exp_values = torch.zeros(n)  # will store unnormalized exp values
    
    print(f"Computing softmax of {n} values in blocks of {block_size}:")
    for i in range(0, n, block_size):
        block = x_full[i:i+block_size]
        m_block = block.max()
        
        # Update max
        m_new = torch.max(m, m_block)
        
        # Rescale previous accumulations
        l = l * torch.exp(m - m_new) + torch.exp(block - m_new).sum()
        
        # Rescale previous exp values
        exp_values[:i] *= torch.exp(m - m_new)
        exp_values[i:i+block_size] = torch.exp(block - m_new)
        
        m = m_new
        print(f"  Block [{i}:{i+block_size}]: m={m.item():.3f}, l={l.item():.3f}")
    
    # Normalize
    online_result = exp_values / l
    
    # Compare with standard softmax
    standard_result = F.softmax(x_full, dim=0)
    
    max_diff = (online_result - standard_result).abs().max().item()
    print(f"\nMax difference from standard softmax: {max_diff:.2e}")
    
    return online_result, standard_result


x = torch.randn(16)
online, standard = online_softmax_demo(x, block_size=4)

## 4. Flash Attention: Tiled Forward Pass

The algorithm processes Q in row blocks and K,V in column blocks:

```
for each Q block (row tile):
    initialize: O = 0, l = 0, m = -inf
    for each K,V block (column tile):
        1. Compute S_tile = Q_tile @ K_tile^T / sqrt(d)
        2. Find block max: m_tile = rowmax(S_tile)
        3. Update running max: m_new = max(m, m_tile)
        4. Rescale: O = O * exp(m - m_new), l = l * exp(m - m_new)
        5. Compute P_tile = exp(S_tile - m_new)
        6. Update: O += P_tile @ V_tile, l += rowsum(P_tile)
        7. m = m_new
    normalize: O = O / l
```

In [ ]:
def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
                            block_size_r: int = 16, block_size_c: int = 16,
                            causal: bool = False) -> torch.Tensor:
    """Simplified Flash Attention forward pass (single head, no batch for clarity).
    
    This implements the tiling strategy of Flash Attention.
    In real implementations, this runs as a fused CUDA kernel on SRAM.
    
    Args:
        Q: (n, d)
        K: (n, d)
        V: (n, d)
        block_size_r: row block size (for Q)
        block_size_c: column block size (for K, V)
    
    Returns:
        O: (n, d)
    """
    n, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    
    # Output accumulator
    O = torch.zeros(n, d, dtype=Q.dtype)
    # Running statistics per row
    l = torch.zeros(n, 1, dtype=Q.dtype)  # sum of exps
    m = torch.full((n, 1), float('-inf'), dtype=Q.dtype)  # row-wise max
    
    # Process Q in row blocks
    for i in range(0, n, block_size_r):
        i_end = min(i + block_size_r, n)
        Q_block = Q[i:i_end]  # (B_r, d)
        
        # Process K,V in column blocks
        kv_end = i_end if causal else n
        for j in range(0, kv_end, block_size_c):
            j_end = min(j + block_size_c, n)
            K_block = K[j:j_end]  # (B_c, d)
            V_block = V[j:j_end]  # (B_c, d)
            
            # Step 1: Compute tile of attention scores
            S_tile = (Q_block @ K_block.T) * scale  # (B_r, B_c) -- small tile, fits in SRAM
            
            # Apply causal mask within tile
            if causal:
                row_idx = torch.arange(i, i_end).unsqueeze(1)
                col_idx = torch.arange(j, j_end).unsqueeze(0)
                causal_mask = row_idx < col_idx
                S_tile = S_tile.masked_fill(causal_mask, float('-inf'))
            
            # Step 2: Block-wise max
            m_tile = S_tile.max(dim=-1, keepdim=True).values  # (B_r, 1)
            
            # Step 3: Update running max
            m_old = m[i:i_end]
            m_new = torch.max(m_old, m_tile)
            
            # Step 4: Rescale old accumulations
            correction = torch.exp(m_old - m_new)
            O[i:i_end] = O[i:i_end] * correction
            l[i:i_end] = l[i:i_end] * correction
            
            # Step 5: Compute attention weights for this tile
            P_tile = torch.exp(S_tile - m_new)  # (B_r, B_c)
            
            # Step 6: Accumulate
            O[i:i_end] = O[i:i_end] + P_tile @ V_block
            l[i:i_end] = l[i:i_end] + P_tile.sum(dim=-1, keepdim=True)
            
            # Step 7: Update max
            m[i:i_end] = m_new
    
    # Final normalization
    O = O / l
    
    return O


# Verify correctness
n, d = 128, 64
Q = torch.randn(n, d)
K = torch.randn(n, d)
V = torch.randn(n, d)

out_standard = standard_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)).squeeze(0)
out_flash = flash_attention_forward(Q, K, V, block_size_r=32, block_size_c=32)

max_diff = (out_standard - out_flash).abs().max().item()
print(f"Max diff between standard and flash attention: {max_diff:.2e}")
print(f"Outputs match: {max_diff < 1e-5}")

In [ ]:
# Verify with causal masking
out_standard_causal = standard_attention(
    Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0), causal=True
).squeeze(0)
out_flash_causal = flash_attention_forward(Q, K, V, block_size_r=32, block_size_c=32, causal=True)

max_diff_causal = (out_standard_causal - out_flash_causal).abs().max().item()
print(f"Causal max diff: {max_diff_causal:.2e}")
print(f"Causal outputs match: {max_diff_causal < 1e-5}")

## 5. Visualize Tiling Strategy

How the attention matrix is processed tile-by-tile:

In [ ]:
def visualize_tiling(n: int = 16, block_r: int = 4, block_c: int = 4, causal: bool = False):
    """Visualize how Flash Attention tiles the attention matrix."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Standard attention: full matrix materialized
    ax = axes[0]
    matrix = np.ones((n, n))
    if causal:
        for i in range(n):
            for j in range(n):
                if j > i:
                    matrix[i][j] = 0
    ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Standard: Full {n}x{n} matrix in HBM', fontsize=12)
    ax.set_xlabel('K positions')
    ax.set_ylabel('Q positions')
    ax.text(n/2, -1.5, f'Memory: O(n^2) = {n*n} elements', ha='center', fontsize=10, color='red')
    
    # Flash attention: tiled processing
    ax = axes[1]
    tile_order = []
    tile_idx = 0
    matrix2 = np.zeros((n, n))
    
    for i in range(0, n, block_r):
        kv_end = min(i + block_r, n) if causal else n
        for j in range(0, kv_end, block_c):
            i_end = min(i + block_r, n)
            j_end = min(j + block_c, n)
            matrix2[i:i_end, j:j_end] = tile_idx + 1
            tile_order.append((i, j, tile_idx))
            tile_idx += 1
    
    cmap = plt.cm.Set3
    ax.imshow(matrix2, cmap=cmap, vmin=0, vmax=tile_idx+1)
    
    # Draw tile boundaries
    for i in range(0, n+1, block_r):
        ax.axhline(y=i-0.5, color='black', linewidth=1)
    for j in range(0, n+1, block_c):
        ax.axvline(x=j-0.5, color='black', linewidth=1)
    
    # Number each tile
    for i, j, idx in tile_order:
        ax.text(j + block_c/2 - 0.5, i + block_r/2 - 0.5, str(idx+1),
                ha='center', va='center', fontsize=10, fontweight='bold')
    
    ax.set_title(f'Flash: {block_r}x{block_c} tiles, processed sequentially', fontsize=12)
    ax.set_xlabel('K positions')
    ax.set_ylabel('Q positions')
    ax.text(n/2, -1.5, f'SRAM: O(B_r * B_c) = {block_r*block_c} elements at a time',
            ha='center', fontsize=10, color='green')
    
    plt.suptitle(f'Attention Matrix Tiling (n={n})' + (' [Causal]' if causal else ''),
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"Total tiles: {tile_idx}")
    print(f"Max SRAM per tile: {block_r * block_c} (scores) + {block_r * 64} (Q) + {block_c * 64} (K,V)")

visualize_tiling(n=16, block_r=4, block_c=4, causal=False)
visualize_tiling(n=16, block_r=4, block_c=4, causal=True)

## 6. Memory-Efficient Backward Pass (Recomputation)

Standard attention backward needs the saved attention matrix $P$ (n x n). Flash Attention instead **recomputes** $P$ from $Q, K$ during the backward pass.

This trades compute for memory: ~25% more FLOPs but $O(n)$ memory instead of $O(n^2)$.

In [ ]:
class MemoryEfficientAttention(torch.autograd.Function):
    """Attention with recomputation in backward pass.
    
    Forward: compute attention output, save only Q, K, V, output, logsumexp
    Backward: recompute attention weights from saved Q, K, V
    """
    @staticmethod
    def forward(ctx, Q, K, V):
        # Standard forward
        d_k = Q.shape[-1]
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Save logsumexp for numerical stability in backward
        m = scores.max(dim=-1, keepdim=True).values
        exp_scores = torch.exp(scores - m)
        l = exp_scores.sum(dim=-1, keepdim=True)
        attn = exp_scores / l
        
        output = attn @ V
        
        # Save for backward -- NOTE: we do NOT save the n x n attention matrix
        logsumexp = m.squeeze(-1) + torch.log(l.squeeze(-1))  # (n,)
        ctx.save_for_backward(Q, K, V, output, logsumexp)
        
        return output
    
    @staticmethod
    def backward(ctx, grad_output):
        Q, K, V, output, logsumexp = ctx.saved_tensors
        d_k = Q.shape[-1]
        
        # Recompute attention weights (the key memory saving)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
        attn = torch.exp(scores - logsumexp.unsqueeze(-1))  # recomputed
        
        # Standard attention backward
        dV = attn.transpose(-2, -1) @ grad_output
        d_attn = grad_output @ V.transpose(-2, -1)
        
        # Softmax backward
        D = (grad_output * output).sum(dim=-1, keepdim=True)
        d_scores = attn * (d_attn - D) / math.sqrt(d_k)
        
        dQ = d_scores @ K
        dK = d_scores.transpose(-2, -1) @ Q
        
        return dQ, dK, dV


# Compare memory usage
n, d = 256, 64
Q = torch.randn(1, n, d, requires_grad=True)
K = torch.randn(1, n, d, requires_grad=True)
V = torch.randn(1, n, d, requires_grad=True)

# Memory-efficient version
out_efficient = MemoryEfficientAttention.apply(Q, K, V)
loss = out_efficient.sum()
loss.backward()

# Standard version for comparison
Q2 = Q.detach().clone().requires_grad_(True)
K2 = K.detach().clone().requires_grad_(True)
V2 = V.detach().clone().requires_grad_(True)
out_standard = standard_attention(Q2, K2, V2)
loss2 = out_standard.sum()
loss2.backward()

print(f"Forward output diff: {(out_efficient - out_standard.detach()).abs().max():.2e}")
print(f"dQ diff: {(Q.grad - Q2.grad).abs().max():.2e}")
print(f"dK diff: {(K.grad - K2.grad).abs().max():.2e}")
print(f"dV diff: {(V.grad - V2.grad).abs().max():.2e}")
print(f"\nMemory saved: attention matrix ({n}x{n} = {n*n} elements) not stored")

## 7. Memory Scaling Analysis

In [ ]:
# Theoretical memory comparison
seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536]
d = 128  # head dimension

standard_memory = []  # O(n^2)
flash_memory = []     # O(n)

for n in seq_lengths:
    # Standard: Q,K,V (3*n*d) + S (n*n) + P (n*n) + O (n*d) = 3nd + 2n^2 + nd
    std_mem = (4 * n * d + 2 * n * n) * 4  # bytes (fp32)
    standard_memory.append(std_mem / 1e9)  # GB
    
    # Flash: Q,K,V (3*n*d) + O (n*d) + logsumexp (n) in HBM
    # Plus SRAM: Q_tile + K_tile + V_tile + S_tile = B_r*d + B_c*d + B_c*d + B_r*B_c
    flash_mem = (4 * n * d + n) * 4  # HBM bytes
    flash_memory.append(flash_mem / 1e9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(seq_lengths, standard_memory, 'r-o', label='Standard Attention', markersize=5)
axes[0].plot(seq_lengths, flash_memory, 'g-s', label='Flash Attention', markersize=5)
axes[0].set_xlabel('Sequence Length')
axes[0].set_ylabel('HBM Memory (GB)')
axes[0].set_title('Memory Scaling (d=128, single head)')
axes[0].legend()
axes[0].set_xscale('log', base=2)
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Memory ratio
ratios = [s / f for s, f in zip(standard_memory, flash_memory)]
axes[1].bar(range(len(seq_lengths)), ratios, color='steelblue', edgecolor='white')
axes[1].set_xticks(range(len(seq_lengths)))
axes[1].set_xticklabels([str(n) for n in seq_lengths], rotation=45)
axes[1].set_xlabel('Sequence Length')
axes[1].set_ylabel('Memory Reduction Factor')
axes[1].set_title('Flash Attention Memory Savings')
for i, r in enumerate(ratios):
    axes[1].text(i, r + max(ratios)*0.02, f'{r:.0f}x', ha='center', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. IO Complexity Analysis

Flash Attention is **IO-aware**: it minimizes reads/writes to HBM.

| | HBM reads | HBM writes | SRAM usage |
|---|-----------|------------|------------|
| Standard | $O(Nd + N^2)$ | $O(Nd + N^2)$ | - |
| Flash Attention | $O(N^2 d^2 / M)$ | $O(Nd)$ | $O(M)$ |

where $M$ = SRAM size, $N$ = sequence length, $d$ = head dimension.

For typical values ($M \gg d^2$), Flash Attention makes $O(N^2 d / M)$ fewer HBM accesses.

In [ ]:
# IO analysis
d = 128  # head dimension
M = 100 * 1024  # SRAM size (~100KB in elements)

seq_lengths_io = [512, 1024, 2048, 4096, 8192]

standard_io = []
flash_io = []

for N in seq_lengths_io:
    # Standard: read Q,K -> write S -> read S -> write P -> read P,V -> write O
    std = 4 * N * d + 2 * N * N  # total HBM reads + writes
    standard_io.append(std)
    
    # Flash: read Q,K,V tiles, write O
    # Number of tiles: (N/B_r) * (N/B_c) where B_r * B_c ~ M
    B_c = min(M // (4 * d), N)  # fit K,V blocks in SRAM
    B_r = min(M // (4 * d), N)  # fit Q block in SRAM  
    n_outer = math.ceil(N / B_r)
    n_inner = math.ceil(N / B_c)
    flash = n_outer * (N * d + n_inner * N * d // n_outer) + N * d  # simplified
    flash_io.append(flash)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(seq_lengths_io))
width = 0.35
ax.bar(x - width/2, [s/1e6 for s in standard_io], width, label='Standard', color='#e74c3c')
ax.bar(x + width/2, [f/1e6 for f in flash_io], width, label='Flash', color='#2ecc71')
ax.set_xticks(x)
ax.set_xticklabels(seq_lengths_io)
ax.set_xlabel('Sequence Length')
ax.set_ylabel('HBM IO (millions of elements)')
ax.set_title('HBM IO Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Block Size Selection

In [ ]:
# Effect of block size on correctness (all should match standard)
n, d = 256, 64
Q = torch.randn(n, d)
K = torch.randn(n, d)
V = torch.randn(n, d)

ref = standard_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)).squeeze(0)

block_sizes = [4, 8, 16, 32, 64, 128, 256]
errors = []
num_tiles = []

for bs in block_sizes:
    out = flash_attention_forward(Q, K, V, block_size_r=bs, block_size_c=bs)
    err = (ref - out).abs().max().item()
    errors.append(err)
    tiles = math.ceil(n / bs) ** 2
    num_tiles.append(tiles)
    print(f"Block size {bs:>4d}: max error = {err:.2e}, tiles = {tiles:>5d}, "
          f"SRAM per tile = {bs*bs + 2*bs*d} elements")

fig, ax1 = plt.subplots(figsize=(10, 5))

color1 = 'steelblue'
ax1.bar(range(len(block_sizes)), num_tiles, color=color1, alpha=0.6)
ax1.set_xticks(range(len(block_sizes)))
ax1.set_xticklabels(block_sizes)
ax1.set_xlabel('Block Size')
ax1.set_ylabel('Number of Tiles', color=color1)

ax2 = ax1.twinx()
sram_per_tile = [bs*bs + 2*bs*d for bs in block_sizes]
ax2.plot(range(len(block_sizes)), sram_per_tile, 'ro-', linewidth=2)
ax2.set_ylabel('SRAM per Tile (elements)', color='red')

plt.title(f'Block Size Tradeoff (n={n}, d={d})')
plt.tight_layout()
plt.show()

print("\nTradeoff: larger blocks = fewer tiles (less overhead) but more SRAM needed.")
print("Block size is chosen to maximize SRAM utilization without overflow.")

## 10. Practical Performance: Wall Clock Comparison

Even in pure Python/PyTorch, we can see the algorithmic behavior.

In [ ]:
# Time comparison at different sequence lengths
# Note: real speedup requires CUDA kernel; this shows algorithmic scaling
seq_lengths_bench = [64, 128, 256, 512, 1024]
d = 64
standard_times = []
flash_times = []

for n in seq_lengths_bench:
    Q = torch.randn(n, d)
    K = torch.randn(n, d)
    V = torch.randn(n, d)
    
    # Standard
    start = time.time()
    for _ in range(5):
        _ = standard_attention(Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0))
    standard_times.append((time.time() - start) / 5)
    
    # Flash (our Python implementation)
    bs = min(64, n)
    start = time.time()
    for _ in range(5):
        _ = flash_attention_forward(Q, K, V, block_size_r=bs, block_size_c=bs)
    flash_times.append((time.time() - start) / 5)

plt.figure(figsize=(10, 5))
plt.plot(seq_lengths_bench, [t*1000 for t in standard_times], 'r-o', label='Standard', markersize=6)
plt.plot(seq_lengths_bench, [t*1000 for t in flash_times], 'g-s', label='Flash (Python)', markersize=6)
plt.xlabel('Sequence Length')
plt.ylabel('Time (ms)')
plt.title('Attention Computation Time\n(Note: real Flash Attention speedup is from fused CUDA kernels)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Note: Our Python Flash Attention is SLOWER because the real benefit comes from")
print("fused CUDA kernels that exploit SRAM locality. The purpose here is to show")
print("the algorithm, not the speed. Real Flash Attention is 2-4x faster than standard.")

## 11. Flash Attention Variants

| Version | Key Improvement |
|---------|----------------|
| Flash Attention 1 (Dao 2022) | Tiled computation, online softmax |
| Flash Attention 2 (Dao 2023) | Better work partitioning across warps, ~2x faster |
| Flash Attention 3 (2024) | Hopper GPU features (TMA, warp specialization) |
| FlashDecoding | Optimized for autoregressive inference (long KV cache) |
| PagedAttention (vLLM) | Virtual memory for KV cache management |

In [ ]:
# Summary comparison table
print("=" * 75)
print(f"{'':>25} {'Standard':>15} {'Flash Attn':>15} {'Ratio':>10}")
print("=" * 75)

for n in [1024, 4096, 16384, 65536]:
    d = 128
    std_mem = (2 * n * n + 4 * n * d) * 2  # fp16 bytes
    flash_mem = (4 * n * d + n) * 2  # fp16 bytes
    
    std_str = f"{std_mem/1e6:.1f} MB" if std_mem < 1e9 else f"{std_mem/1e9:.1f} GB"
    flash_str = f"{flash_mem/1e6:.1f} MB" if flash_mem < 1e9 else f"{flash_mem/1e9:.1f} GB"
    
    print(f"n={n:>6d}, d={d:>3d}:  {std_str:>15} {flash_str:>15} {std_mem/flash_mem:>8.0f}x")

print("=" * 75)
print("\nFlash Attention enables context lengths of 100K+ tokens that would be")
print("impossible with standard attention due to memory constraints.")

## Interview Notes

**"Why is standard attention memory-inefficient?"**
- It materializes the full $N \times N$ attention matrix ($QK^T$ and softmax output) in HBM
- Memory scales quadratically: $O(N^2)$ for the attention matrix vs $O(Nd)$ for Q,K,V,O
- At N=8192, d=128: attention matrix is 512MB, while Q,K,V are only 12MB total
- The matrix is read/written multiple times: compute -> softmax -> dropout -> matmul with V

**"How does Flash Attention avoid materializing the full attention matrix?"**
- Tiles Q into row blocks and K,V into column blocks
- Each tile computes a small block of the attention matrix in fast SRAM
- Uses the **online softmax trick** to incrementally accumulate the correct softmax result
- Only the final output O is written back to HBM -- never the full attention matrix
- The result is **exact** (not an approximation)

**"What is the online softmax trick?"**
- Softmax normally requires two passes: (1) find max, (2) compute exp/sum
- Online softmax maintains running max and running sum as new blocks are processed
- When a new block has a larger max, rescale previous accumulations: multiply by $e^{m_{\text{old}} - m_{\text{new}}}$
- This allows processing softmax incrementally without seeing all values

**"SRAM vs HBM?"**
- HBM: High Bandwidth Memory, ~40-80GB, ~1.5-3 TB/s bandwidth
- SRAM: On-chip memory, ~20MB on A100, ~19 TB/s bandwidth (10-20x faster)
- Flash Attention is IO-bound optimization: reduces HBM reads/writes
- Same total FLOPs (slightly more due to recomputation), but much less IO

**"Backward pass?"**
- Standard backward saves the full attention matrix P for the backward pass
- Flash Attention saves only logsumexp statistics (O(N) memory)
- Recomputes P tile-by-tile during backward -- extra FLOPs but massive memory saving
- Enables training with much longer sequences on the same hardware